In [ ]:
# Package installs go here 
%conda install polars, plotly

/opt/anaconda3/lib/python3.12/site-packages/conda/base/context.py:201: FutureWarning: Adding 'defaults' to channel list implicitly is deprecated and will be removed in 25.3. 

To remove this warning, please choose a default channel explicitly with conda's regular configuration system, e.g. by adding 'defaults' to the list of channels:

  conda config --add channels defaults

For more information see https://docs.conda.io/projects/conda/en/stable/user-guide/configuration/use-condarc.html

  deprecated.topic(
/opt/anaconda3/lib/python3.12/site-packages/conda/base/context.py:201: FutureWarning: Adding 'defaults' to channel list implicitly is deprecated and will be removed in 25.3. 

To remove this warning, please choose a default channel explicitly with conda's regular configuration system, e.g. by adding 'defaults' to the list of channels:

  conda config --add channels defaults

For more information see https://docs.conda.io/projects/conda/en/stable/user-guide/configuration/use-condarc.

In [3]:
# python imports 
import polars as pl
import plotly.io as pio
# VS Code draws the plotly mimetype natively (fast); the HTML copy keeps charts in the slides export
pio.renderers.default = "plotly_mimetype+notebook_connected"

In [ ]:
# Read eBird csv data
# ebird = pl.read_csv("data/ebird.csv") # Don't need ebird data 
observations = pl.read_csv("data/observation.csv") 
species = pl.read_csv("data/899_Species.csv") 
header = pl.read_csv(
    "data/header.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
    schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
)
locations = pl.read_csv(
    "data/location.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
    schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
)

traits = pl.read_csv(
    "data/traits.csv",
    null_values=["NA", ""],
    infer_schema_length=100_000,
)
synonyms = pl.read_csv("data/synonyms.csv") 


In [58]:
# Helper function that converts form degree:minutes to degree:decimal
def ddmm_to_dd(column_name: str) -> pl.Expr:
    degree = pl.col(column_name) // 100
    minutes = pl.col(column_name) - degree * 100
    return degree + (minutes / 60)

ebird = ebird.with_columns(
    key_latitude = -ddmm_to_dd("key_latitude"), 
    key_longitude = ddmm_to_dd("key_longitude")
)

ebird = ebird.with_columns(
    latitude = -ddmm_to_dd("latitude"), 
    longitude = ddmm_to_dd("longitude")
)




In [59]:
# Change ebird start and end dates from string to date type (format is YYYY/MM/DD)
ebird = ebird.with_columns(
    pl.col("Observation Start Date", "Observation End Date").str.to_date("%Y/%m/%d")
)

## eBird heatmap

In [12]:
# Code for eBird Heatmap
import plotly.express as px

# NSW extent (same bounds as the R notebook)
NSW_LAT = (-37.7, -28.0)
NSW_LON = (140.5, 164.0)

# eBird key coordinates are the centres of 10-minute Bird Atlas grid cells,
# so each tile reaches 5 minutes either side of its centre
HALF_CELL = 5 / 60

# Count observations in each grid cell
nsw_cells = (
    ebird
    .filter(
        pl.col("key_latitude").is_between(*NSW_LAT),
        pl.col("key_longitude").is_between(*NSW_LON),
    )
    .group_by("key_latitude", "key_longitude")
    .len(name="Observations")
    .with_row_index("cell_id")
    .with_columns(log_observations=pl.col("Observations").log10())
)

# Turn each grid cell into a square polygon so plotly can draw it as a tile
def cell_square(lat, lon):
    return [[
        [lon - HALF_CELL, lat - HALF_CELL],
        [lon + HALF_CELL, lat - HALF_CELL],
        [lon + HALF_CELL, lat + HALF_CELL],
        [lon - HALF_CELL, lat + HALF_CELL],
        [lon - HALF_CELL, lat - HALF_CELL],
    ]]

grid_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "id": row["cell_id"],
            "geometry": {"type": "Polygon", "coordinates": cell_square(row["key_latitude"], row["key_longitude"])},
        }
        for row in nsw_cells.iter_rows(named=True)
    ],
}

# Plasma runs yellow -> orange -> pink -> purple -> deep blue, changing hue as well as
# lightness so neighbouring values are easy to tell apart. It is also colourblind-safe.
# Reversed so yellow = few observations and deep blue = many.
COLOUR_SCALE = px.colors.sequential.Plasma_r

# Counts range from 1 to over 100,000, so colour by log10 and label the scale in real counts
max_log = nsw_cells["log_observations"].max()
tick_values = list(range(0, int(max_log) + 1))
tick_labels = [f"{10 ** v:,}" for v in tick_values]

fig = px.choropleth_mapbox(
    nsw_cells,
    geojson=grid_geojson,
    locations="cell_id",
    color="log_observations",
    color_continuous_scale=COLOUR_SCALE,
    range_color=(0, max_log),
    custom_data=["Observations", "key_latitude", "key_longitude"],
    mapbox_style="open-street-map",
    center={"lat": -32.8, "lon": 147.0},
    zoom=4.6,
    opacity=0.85,
    title="Heatmap of eBird observations in NSW",
)
fig.update_traces(
    marker_line_width=0,
    hovertemplate="%{customdata[0]:,} observations<br>Cell centre: %{customdata[1]:.2f}, %{customdata[2]:.2f}<extra></extra>",
)
fig.update_layout(
    height=650,
    margin=dict(l=0, r=0, t=50, b=0),
    coloraxis_colorbar=dict(title="Observations", tickvals=tick_values, ticktext=tick_labels),
)
fig.show()

## DES Notes 
- Number of observations in the DES Data  
- As this data is more recent, compare reporting rates to pre-2020 surveys 
- Are there any previously reported species missing in the DES data. 
- Where are the DES reports concentrated. 
- Isolate and analyze interesting / popular species 
- The des data has region codes
- It's an xslx file but the first table has everything we need to know about each survey. 
- 



### Regions 
- There are 9 regions in the dataset identified by a code.  
- What are the significance of these regions?  
- Rename the regions from codes to their actual names. 

In [62]:
# Reading DES data: only the first sheet of the spreadsheet
# engine="openpyxl" because polars' default Excel reader (fastexcel) isn't installed
des = pl.read_excel(
    "data/des.xlsx",
    sheet_name="DES_Backlog_Format",
    engine="openpyxl",
)
des

Survey ID,DES Ref,Latitude,Longitude,Observation Start Date,Observation End Date,Locality,Comments,Collection flag,Observer Id,Observer Last Name,Observer First Name,Location name,Region code,Species Common Name,Species ID,Breeding Code,Quality flag,Species Comments [Notes],K-Latitude,K-Longitude
i64,str,i64,i64,datetime[μs],datetime[μs],str,str,str,str,str,str,str,str,str,str,str,null,str,str,str
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Emu""","""1""","""Y""",null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Masked Lapwing""","""133""","""Y""",null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Australian Wood Duck""","""202""",null,null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Pacific Black Duck""","""208""",null,null,"""NULL""","""2945""","""15035"""
900000,"""BPC46-26009.028""",2945,15035,2025-08-01 00:00:00,2025-08-31 00:00:00,"""Mount Rodd""","""NULL""","""NULL""","""26009""","""Swain""","""Bob""","""Mount Rodd""","""WS""","""Australasian Grebe""","""61""",null,null,"""NULL""","""2945""","""15035"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Stubble Quail""","""9""",null,null,"""NULL""","""3155""","""14925"""
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Australian Magpie""","""705""",null,null,"""NULL""","""3155""","""14925"""
902577,"""BPC18-26044.033""",3155,14925,2022-08-21 00:00:00,2022-08-21 00:00:00,"""Moggymill""","""NULL""","""NULL""","""26044""","""Goulder""","""Graham""","""Moggymill""","""WS""","""Willie Wagtail""","""364""",null,null,"""NULL""","""3155""","""14925"""


In [67]:
# Converts DES coordinates from degrees-minutes (DDMM) to decimal degrees
# The K- columns are a cleaned copy of Latitude/Longitude: they fix small typos
# (e.g. 15034 -> 15035) and hold the text "NULL" where the original was unusable.
des = des.rename({
    "K-Latitude": "key_latitude", "K-Longitude": "key_longitude",
    "Latitude": "latitude", "Longitude": "longitude",
})

# Turn the text "NULL" into a real missing value, then make the key columns numbers
des = des.with_columns(
    pl.col("key_latitude", "key_longitude").replace("NULL", None).cast(pl.Int64)
)

# A DDMM value is only a real coordinate if its minutes are below 60 and its degrees
# are inside Australia's range. Anything else (e.g. 30015 or 15275) becomes missing.
def valid_ddmm(column_name: str, min_degree: int, max_degree: int) -> pl.Expr:
    degree = pl.col(column_name) // 100
    minutes = pl.col(column_name) % 100
    return (minutes < 60) & degree.is_between(min_degree, max_degree)

des = des.with_columns(
    pl.when(valid_ddmm(c, 9, 45)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["latitude", "key_latitude"]
).with_columns(
    pl.when(valid_ddmm(c, 112, 160)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["longitude", "key_longitude"]
)

# Same conversion as for eBird: latitude is south, so it becomes negative
des = des.with_columns(
    key_latitude=-ddmm_to_dd("key_latitude"),
    key_longitude=ddmm_to_dd("key_longitude"),
    latitude=-ddmm_to_dd("latitude"),
    longitude=ddmm_to_dd("longitude"),
)

des.select("latitude", "longitude", "key_latitude", "key_longitude").describe()

statistic,latitude,longitude,key_latitude,key_longitude
str,f64,f64,f64,f64
"""count""",104103.0,104125.0,104103.0,104125.0
"""null_count""",37.0,15.0,37.0,15.0
"""mean""",-31.436353,151.045275,-31.436357,151.045286
"""std""",1.804477,2.290407,1.804481,2.290405
"""min""",-37.416667,141.083333,-37.416667,141.083333
"""25%""",-33.25,149.916667,-33.25,149.916667
"""50%""",-31.083333,151.416667,-31.083333,151.416667
"""75%""",-30.083333,152.916667,-30.083333,152.916667
"""max""",-21.083333,153.583333,-21.083333,153.583333


In [63]:
# Number of regions in the dataset - 9 Regions  

des_sighting_locations = des['Region code'].value_counts(sort = True)  


region_names = {"NC": "Northern Coast", 
                "RP": "Riverine Plane", 
                "A": "Arid", 
                "CSC": "Central and South Coast", 
                "WS": "Western Slopes", 
                "T": "Tablelands", 
                "MAW": "Malee and Acacia Woodlands", 
                "CP": "Central Peneplain",
                "TS": "Tasman Sea"
                }

In [66]:
# DES heatmap: number of observations in each 10-minute grid cell across NSW

# Keep only locations inside NSW (NSW_LAT and NSW_LON are defined in the eBird heatmap cell)
des_nsw = des.filter(
    pl.col("key_latitude").is_between(*NSW_LAT),
    pl.col("key_longitude").is_between(*NSW_LON),
)

# Frequency table of locations, with an id for each grid cell
location_counts = (
    des_nsw
    .select("key_latitude", "key_longitude")
    .drop_nulls()
    .group_by("key_latitude", "key_longitude")
    .len("observations")
    # len() gives an unsigned integer, which plotly mistakes for categories
    # (one layer per distinct count). Casting to Int64 keeps it numeric.
    .with_columns(pl.col("observations").cast(pl.Int64))
    .with_row_index("cell_id")
)

# Put each cell into a logarithmic band. left_closed=True puts exactly 10 in "10–99", not "1–9".
BAND_LABELS = ["1–9", "10–99", "100–999", "1,000–9,999", "10,000+"]
location_counts = location_counts.with_columns(
    band=pl.col("observations").cut(
        breaks=[10, 100, 1000, 10000], labels=BAND_LABELS, left_closed=True
    )
)

# Each key coordinate is the centre of a 10-minute square, so h is half its width
h = 5 / 60

def square(lat, lon):
    return [[[lon - h, lat - h], [lon + h, lat - h],
             [lon + h, lat + h], [lon - h, lat + h], [lon - h, lat - h]]]

shapes = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "id": row["cell_id"],
         "geometry": {"type": "Polygon", "coordinates": square(row["key_latitude"], row["key_longitude"])}}
        for row in location_counts.iter_rows(named=True)
    ],
}

band_colours = dict(zip(BAND_LABELS, px.colors.sample_colorscale("Plasma_r", 5)))

fig = px.choropleth_mapbox(
    location_counts, geojson=shapes, locations="cell_id", color="band",
    color_discrete_map=band_colours,
    category_orders={"band": BAND_LABELS},   # legend runs from fewest to most
    custom_data=["observations"],
    mapbox_style="open-street-map", center={"lat": -32.8, "lon": 147.0}, zoom=4.6,
    opacity=0.85,
    title="Heatmap of DES observations in NSW: 2020 - 2026",
)
fig.update_traces(marker_line_width=0, hovertemplate="%{customdata[0]:,} observations<extra></extra>")
fig.update_layout(height=650, margin=dict(l=0, r=0, t=50, b=0), legend_title_text="Observations")
fig.show()

NameError: name 'NSW_LAT' is not defined

## Observations heatmap
The observations table has no coordinates, so each sighting takes its location from the survey it belongs to (`sd_ID` in observations matches `s_ID` in the survey headers).

In [47]:
# Observations heatmap: number of sightings in each 10-minute grid cell across NSW

# Attach each sighting to its survey to get the survey's grid-cell coordinates.
# An inner join drops the ~1,200 sightings whose survey isn't in the header table.
obs_with_location = (
    observations
    .join(header.select("s_ID", "k_Latitude", "k_Longitude"), left_on="sd_ID", right_on="s_ID")
    .rename({"k_Latitude": "key_latitude", "k_Longitude": "key_longitude"})
    # Header coordinates are DDMM like eBird's, and latitude is south
    .with_columns(
        key_latitude=-ddmm_to_dd("key_latitude"),
        key_longitude=ddmm_to_dd("key_longitude"),
    )
)

# Count sightings per grid cell inside NSW (a handful of typo'd cells fall outside and drop out here)
obs_counts = (
    obs_with_location
    .filter(
        pl.col("key_latitude").is_between(*NSW_LAT),
        pl.col("key_longitude").is_between(*NSW_LON),
    )
    .group_by("key_latitude", "key_longitude")
    .len("observations")
    .with_columns(pl.col("observations").cast(pl.Int64))   # keep plotly treating it as numeric
    .with_row_index("cell_id")
)

# Busiest cells pass 100,000 sightings, so this needs two more bands than the DES map
OBS_BAND_LABELS = ["1–9", "10–99", "100–999", "1,000–9,999", "10,000–99,999", "100,000+"]
obs_counts = obs_counts.with_columns(
    band=pl.col("observations").cut(
        breaks=[10, 100, 1000, 10000, 100000], labels=OBS_BAND_LABELS, left_closed=True
    )
)

# Grid-cell squares, using square() from the DES heatmap cell
obs_shapes = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "id": row["cell_id"],
         "geometry": {"type": "Polygon", "coordinates": square(row["key_latitude"], row["key_longitude"])}}
        for row in obs_counts.iter_rows(named=True)
    ],
}

obs_band_colours = dict(zip(OBS_BAND_LABELS, px.colors.sample_colorscale("Plasma_r", len(OBS_BAND_LABELS))))

fig = px.choropleth_mapbox(
    obs_counts, geojson=obs_shapes, locations="cell_id", color="band",
    color_discrete_map=obs_band_colours,
    category_orders={"band": OBS_BAND_LABELS},
    custom_data=["observations"],
    mapbox_style="open-street-map", center={"lat": -32.8, "lon": 147.0}, zoom=4.6,
    opacity=0.85,
    title="Heatmap of observations in NSW from 1960 - 2019",
)
fig.update_traces(marker_line_width=0, hovertemplate="%{customdata[0]:,} observations<extra></extra>")
fig.update_layout(height=650, margin=dict(l=0, r=0, t=50, b=0), legend_title_text="Observations")
fig.show()

NameError: name 'NSW_LAT' is not defined

# Ragav's Research questions  

- Spatial range-shift type maps. Such as where the Rock Warbler was found in 1960 against where it was found in 2010. 

    - One graph that shows all spottings of Rock warlblers in 1960 juxtaposed with where it was found in the 2010. 

    - What has changed in the last 50 years? 

- Are bird species consistenly found in one region or are they found in multiple regions?  

- Enumerate the number of unique bird species in one region for each decade from 1960 - 2010.

- Closely Observe patterns of endangered bird species. 
    -  Obtain a list of endangered bird species from the given dataset. 
    - Trends 
        - Is the reporting rate of these bird species declining? 
        - Number of habitats declining? 
        - How varied are the diets. 
    - If something showing an increasing or declining pattern. Is there any co-variate responsible for this? 

- Classify species into two groups: Declining species and Increasing species. 
    - Were any "increasing" species previosly present in the "declining" group and vice versa? 
    - Is there anything trait (Habitat, diet, region) common about declining species?    
        - Do further research on why that particular trait is in endangered? 

- Association between rainfall patterns / habitat destruction. 
    - Habitat destructions in NSW. 
    - Rainfall patterns in NSW
        - Look at the rainfall across trends in NSW.
            - Dry regions
            - Low rainfall 
            - Moderate rainfall
            - High rainfall
        








## Endangered Species 

1. King Island Brown Thornbill
    - **Scientefic Name:**   
    - **Bird type:** *small* insect eating bird 






In [ ]:
# import polars as pl

# # Read eBird csv data
# ebird = pl.read_csv("data/ebird.csv") 
# observations = pl.read_csv("data/observation.csv") 
# species = pl.read_csv("data/899_Species.csv") 
# header = pl.read_csv(
#     "data/header.csv",
#     null_values=["NA", ""],
#     infer_schema_length=100_000,
#     schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
# )
# locations = pl.read_csv(
#     "data/location.csv",
#     null_values=["NA", ""],
#     infer_schema_length=100_000,
#     schema_overrides={"k_Latitude": pl.Float64, "k_Longitude": pl.Float64},
# )

# traits = pl.read_csv(
#     "data/traits.csv",
#     null_values=["NA", ""],
#     infer_schema_length=100_000,
# )
# synonyms = pl.read_csv("data/synonyms.csv")

In [ ]:
# Data table view
traits

ID,Scientific_Name,Common_Name,Order_Name,Family_Name,ALA_IUCN,Local_Status,Population,Species_Type,Primary_Diet,ALA_Diet_Code,Diet_Breadth,ESI,Foraging_Strategy,Primary_Habitat,Habitat_Breadth,Nest_Type,Nest_Substrate,Cooperative_Breeder,Incubating_Sex,Social_Behaviour,Monogamy,Mobility,Body_Mass_g,Body_Size,Exclusion_Zone,Elev_NormMin,Elev_NormMax,breeding_season_start,breeding_season_end,breeding_season_notes
f64,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,f64,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str
1.0,"""Struthio camelus""","""Common Ostrich""","""Struthioniformes""","""Struthionidae""","""LC""","""Exotic""","""Introduced""","""Bird""","""Herbivore""","""Plant""",4.0,0.8,"""On ground""","""Savanna""",4.0,"""Scrape""","""Ground""","""Cooperative breeder""","""Both""","""Various""","""Poly""","""Various""",114500.0,"""Very Large""",null,"""0""","""3000""",null,null,null
26.0,"""Casuarius casuarius""","""Southern Cassowary""","""Casuariiformes""","""Casuariidae""","""LC""","""Native""","""Australian""","""Bird""","""Fruit""","""Fruit""",3.0,1.05,"""On ground""","""Forest""",3.0,"""Platform""","""Ground""","""Solitary breeder""","""Male""","""Various""","""Poly""","""Various""",46700.0,"""Very Large""",null,"""0""","""500""",null,null,null
31.0,"""Dromaius novaehollandiae""","""Common Emu""","""Casuariiformes""","""Casuariidae""","""LC""","""Native""","""Endemic""","""Bird""","""Omnivore""","""Fruit""",5.0,0.6,"""Understorey""","""Savanna""",5.0,"""Scrape""","""Ground""",null,"""Male""","""Single/Pairs""","""Mono""","""Various""",42500.0,"""Very Large""",null,"""0""","""1600""",null,null,null
227.0,"""Anseranas semipalmata""","""Magpie Goose""","""Anseriformes""","""Anseranatidae""","""LC""","""Native""","""Australian""","""Bird""","""Plant""","""Plant""",2.0,1.1,"""On ground""","""Wetland""",4.0,"""Cup/Bowl""","""Water""",null,"""Both""","""Territorial""","""Poly""","""Nomadic""",2302.0,"""Very Large""",null,"""0""","""500""",null,null,null
232.0,"""Dendrocygna guttata""","""Spotted Whistling-duck""","""Anseriformes""","""Anatidae""","""LC""","""Native""","""Australian""","""Bird""","""Omnivore""","""Seed""",2.0,1.1,"""Water surface""","""Wetland""",4.0,"""Cavity (Tree)""","""Tree""",null,null,null,null,"""Sedentary""",677.5,"""Large""",null,"""0""","""L""",null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
34913.0,"""Emberiza citrinella""","""Yellowhammer""","""Passeriformes""","""Emberizidae""","""LC""","""Vagrant""","""Vagrant (from NZ introduction)""","""Bird""","""Seed""","""Seed""",2.0,1.1,"""On ground""","""Woodland""",4.0,"""Cup/Bowl""","""Ground, Shrub/Bush/Vine""",null,"""Female""",null,"""Mono""","""Sedentary""",29.73,"""Small""",null,"""600""","""900""",null,null,null
35001.0,"""Emberiza aureola""","""Yellow-breasted Bunting""","""Passeriformes""","""Emberizidae""","""CR""",null,null,"""Bird""","""Omnivore""",null,3.0,0.74,null,"""Grassland""",6.0,"""Cup/Bowl""","""Ground, Shrub/Bush/Vine, Tree""",null,"""Both""",null,"""Mono""","""Migratory""",22.2,"""Small""",null,"""0""","""1000""",null,null,null
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [22]:
endangered_species = traits.filter(pl.col("ALA_IUCN").str.contains("EN"))  
endangered_species

ID,Scientific_Name,Common_Name,Order_Name,Family_Name,ALA_IUCN,Local_Status,Population,Species_Type,Primary_Diet,ALA_Diet_Code,Diet_Breadth,ESI,Foraging_Strategy,Primary_Habitat,Habitat_Breadth,Nest_Type,Nest_Substrate,Cooperative_Breeder,Incubating_Sex,Social_Behaviour,Monogamy,Mobility,Body_Mass_g,Body_Size,Exclusion_Zone,Elev_NormMin,Elev_NormMax,breeding_season_start,breeding_season_end,breeding_season_notes
f64,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,f64,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str
6616.0,"""Rostratula australis""","""Australian Painted-snipe""","""Charadriiformes""","""Rostratulidae""","""EN""","""Native""","""Endemic""","""Bird""","""Invertebrate""","""Invertebrates""",3.0,1.22,"""On ground""","""Artificial""",2.0,"""Scrape""","""Ground""",null,"""Male""","""Various""","""Poly""","""Nomadic""",128.0,"""Large""",null,"""0""","""700""",null,null,null
6639.0,"""Pedionomus torquatus""","""Plains-wanderer""","""Charadriiformes""","""Pedionomidae""","""EN""","""Native""","""Endemic""","""Bird""","""Omnivore""","""Invertebrates""",2.0,1.4,"""On ground""","""Grassland""",2.0,"""Scrape""","""Ground""",null,"""Male""","""Solitary""","""Poly""","""Sedentary""",67.25,"""Medium""",null,"""0""","""700""",null,null,null
6672.0,"""Numenius madagascariensis""","""Far Eastern Curlew""","""Charadriiformes""","""Scolopacidae""","""EN""","""Exotic""","""Non-breeding""","""Bird""","""Invertebrate""","""Invertebrates""",2.0,1.22,"""On ground""","""Wetland""",3.0,"""Mound""","""Ground""",null,null,"""Territorial""",null,"""Migratory""",815.0,"""Large""",null,"""0""","""500""",null,null,null
6783.0,"""Tringa guttifer""","""Spotted Greenshank""","""Charadriiformes""","""Scolopacidae""","""EN""","""Vagrant""","""Vagrant""","""Bird""","""Carnivore""","""Invertebrates""",2.0,0.92,"""On ground""","""Wetland""",6.0,null,"""Tree""",null,"""Both""","""Various""","""Mono""","""Migratory""",146.0,"""Large""",null,"""0""","""500""",null,null,null
6799.0,"""Calidris tenuirostris""","""Great Knot""","""Charadriiformes""","""Scolopacidae""","""EN""","""Exotic""","""Non-breeding""","""Bird""","""Invertebrate""","""Invertebrates""",3.0,0.92,"""On ground""","""Grassland""",4.0,null,null,null,"""Both""",null,"""Mono""","""Migratory""",181.0,"""Large""",null,"""0""","""1700""",null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
19522.0,"""Dasyornis longirostris""","""Western Bristlebird""","""Passeriformes""","""Dasyornithidae""","""EN""","""Native""","""Endemic""","""Bird""","""Invertebrate""","""Seed""",2.0,1.7,"""On ground""","""Shrub""",1.0,"""Dome/Oven""","""Grass""",null,null,null,"""Mono""","""Various""",34.0,"""Small""",null,"""0""","""50""",null,null,null
19534.0,"""Pardalotus quadragintus""","""Forty-spotted Pardalote""","""Passeriformes""","""Pardalotidae""","""EN""","""Native""","""Endemic""","""Bird""","""Invertebrate""","""Invertebrates""",2.0,1.7,"""Canopy""","""Forest""",1.0,"""Dome/Oven""","""Stump, Ground, Tree""",null,"""Both""","""Various""","""Mono""","""Various""",11.0,"""Small""",null,"""0""","""1000""",null,null,null
19561.0,"""Oreoscopus gutturalis""","""Fernwren""","""Passeriformes""","""Acanthizidae""","""EN""","""Native""","""Endemic""","""Bird""","""Invertebrate""","""Invertebrates""",2.0,1.7,"""On ground""","""Forest""",1.0,"""Dome/Oven""","""Stump, Ground, Shrub/Bush/Vine""","""Solitary breeder""","""Female""","""Single/Pairs""",null,"""Various""",18.2,"""Small""",null,"""450""","""1500""",null,null,null


In [23]:
#
des = pl.read_excel(
    "data/des.xlsx",
    sheet_name="DES_Backlog_Format",
    engine="openpyxl",
) 



In [ ]:
header

s_ID,s_Atlasser,s_StartF,s_DateL,s_Latitude,s_Longitude,s_Locality,s_Qualifier,s_Notes,s_DESref,s_AREVref,k_Latitude,k_Longitude,observation_Count,confidence_Level,is_Collection
i64,i64,str,str,str,str,str,str,str,str,str,f64,f64,i64,i64,f64
3,23531,"""2005-04-09""","""2005-04-09""","""3255""","""15125""","""Freemans Waterhole""",null,"""ex C Driscoll""",null,null,3255.0,15125.0,1,0,1.0
4,23531,"""2005-04-09""","""2005-04-09""","""3345""","""15105""","""Turramurra""",null,"""ex T Wilson""",null,null,3345.0,15105.0,1,0,1.0
5,23531,"""2005-04-10""","""2005-04-10""","""3355""","""15115""","""Centennial Park""",null,"""ex K Hemming""",null,null,3355.0,15115.0,1,0,1.0
6,23531,"""2005-03-26""","""2005-03-26""","""3345""","""15105""","""East Killara""",null,"""Ex K Lindsay""",null,null,3345.0,15105.0,1,0,1.0
7,23531,"""2005-04-09""","""2005-04-09""","""2945""","""15255""","""Cowans Ponds""",null,"""Ex G Clancy""",null,null,2945.0,15255.0,1,0,1.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
595762,60003,"""2000-02-26""","""2000-03-04""","""3132""","""15904""","""Lord Howe Island""",null,null,"""BA2-3720""",null,3135.0,15905.0,30,2,null
595763,60003,"""2000-10-21""","""2000-10-22""","""3135""","""15905""","""Lord Howe Island""",null,null,"""BA2-5484""",null,3135.0,15905.0,31,0,null
595764,60003,"""2000-09-12""","""2000-11-22""","""3133""","""15905""","""Capella South Lord Howe Island""",null,null,"""BA2-5428""",null,3135.0,15905.0,1,1,1.0


ValueError: 'left_on' requires corresponding 'right_on'

In [64]:
# Join observations -> header (survey) -> species, with DES-style column names
# so the result can be stacked/joined with the cleaned `des` table later.

# Header lat/lon are DDMM text ("-" where missing); key coords are DDMM numbers.
# Reuses valid_ddmm() and ddmm_to_dd() from the DES cleaning cell.
survey_locations = (
    header
    .select(
        pl.col("s_ID").alias("Survey ID"),
        pl.col("s_Latitude").cast(pl.Int64, strict=False).alias("latitude"),
        pl.col("s_Longitude").cast(pl.Int64, strict=False).alias("longitude"),
        pl.col("k_Latitude").cast(pl.Int64).alias("key_latitude"),
        pl.col("k_Longitude").cast(pl.Int64).alias("key_longitude"),
        pl.col("s_Locality").alias("Locality"),
        # strict=False turns the odd invalid date (e.g. "1900-01-00") into null
        pl.col("s_StartF").str.to_date("%Y-%m-%d", strict=False).alias("Observation Start Date"),
        pl.col("s_DateL").str.to_date("%Y-%m-%d", strict=False).alias("Observation End Date"),
    )
)

# The header has no region code, but every 10-minute grid cell in the location
# table belongs to exactly one region, so look it up by the survey's key cell
cell_regions = (
    locations
    .group_by("bl_Lat", "bl_Lon")
    .agg(pl.col("region_Code").first().alias("Region code"))
)
survey_locations = survey_locations.join(
    cell_regions,
    left_on=["key_latitude", "key_longitude"],
    right_on=["bl_Lat", "bl_Lon"],
    how="left",
)

# Same cleaning and DDMM -> decimal degrees conversion as the DES table
survey_locations = survey_locations.with_columns(
    pl.when(valid_ddmm(c, 9, 45)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["latitude", "key_latitude"]
).with_columns(
    pl.when(valid_ddmm(c, 112, 160)).then(pl.col(c)).otherwise(None).alias(c)
    for c in ["longitude", "key_longitude"]
).with_columns(
    key_latitude=-ddmm_to_dd("key_latitude"),
    key_longitude=ddmm_to_dd("key_longitude"),
    latitude=-ddmm_to_dd("latitude"),
    longitude=ddmm_to_dd("longitude"),
)

species_names = species.select(
    pl.col("spNo").alias("Species ID"),
    pl.col("sName").alias("Species Common Name"),
    pl.col("sciName").alias("Species Scientific Name"),
)

# Inner join to the header drops the ~1,000 sightings with no survey (no location);
# left join to species keeps the 22 sightings whose species number isn't in the list
atlas_obs = (
    observations
    .select(pl.col("sd_ID").alias("Survey ID"), pl.col("sd_SpeciesORIG").alias("Species ID"))
    .join(survey_locations, on="Survey ID", how="inner")
    .join(species_names, on="Species ID", how="left")
    .select(
        "Survey ID", "Observation Start Date", "Observation End Date",
        "Species ID", "Species Common Name", "Species Scientific Name",
        "latitude", "longitude", "key_latitude", "key_longitude", "Locality", "Region code",
    )
)
atlas_obs

Survey ID,Observation Start Date,Observation End Date,Species ID,Species Common Name,Species Scientific Name,latitude,longitude,key_latitude,key_longitude,Locality,Region code
i64,date,date,i64,str,str,f64,f64,f64,f64,str,str
241376,2015-10-25,2015-10-31,58,"""Purple Swamphen""","""Porphyrio porphyrio""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS"""
241376,2015-10-25,2015-10-31,137,"""Pacific Golden Plover""","""Pluvialis fulva""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS"""
241376,2015-10-25,2015-10-31,129,"""Ruddy Turnstone""","""Arenaria interpres""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS"""
241376,2015-10-25,2015-10-31,153,"""Bar-tailed Godwit""","""Limosa lapponica""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS"""
241376,2015-10-25,2015-10-31,124,"""Black Noddy""","""Anous minutus""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS"""
…,…,…,…,…,…,…,…,…,…,…,…
595767,2001-03-13,2001-03-13,122,"""Common Noddy""","""Anous stolidus""",-31.75,159.25,-31.75,159.25,"""Balls Pyramid""","""TS"""
595767,2001-03-13,2001-03-13,124,"""Black Noddy""","""Anous minutus""",-31.75,159.25,-31.75,159.25,"""Balls Pyramid""","""TS"""
595767,2001-03-13,2001-03-13,922,"""Kermadec Petrel""","""Pterodroma neglecta""",-31.75,159.25,-31.75,159.25,"""Balls Pyramid""","""TS"""


In [54]:
atlas_obs.columns

['Survey ID',
 'Species ID',
 'Species Common Name',
 'Species Scientific Name',
 'latitude',
 'longitude',
 'key_latitude',
 'key_longitude',
 'Locality',
 'Region code']

In [68]:
# Combine the Atlas sightings (up to 2020) with the DES sightings (2020 - 2026).
# DES survey IDs start at 900000 and Atlas IDs stop below 600000, so stacking the
# two tables can't mix up surveys. "Source" records which dataset each row came from.

# Bring the cleaned DES table into the same shape as atlas_obs
des_obs = (
    des
    .with_columns(
        # DES writes missing values as the text "NULL"
        pl.col("Locality", "Region code").replace("NULL", None),
        pl.col("Species ID").cast(pl.Int64, strict=False),
        pl.col("Observation Start Date", "Observation End Date").cast(pl.Date),
    )
    # DES has common names but no scientific names, so take those from the species list
    .join(species_names.select("Species ID", "Species Scientific Name"), on="Species ID", how="left")
    .select(atlas_obs.columns)
)

all_obs = pl.concat([
    atlas_obs.with_columns(Source=pl.lit("Atlas")),
    des_obs.with_columns(Source=pl.lit("DES")),
])
all_obs

Survey ID,Observation Start Date,Observation End Date,Species ID,Species Common Name,Species Scientific Name,latitude,longitude,key_latitude,key_longitude,Locality,Region code,Source
i64,date,date,i64,str,str,f64,f64,f64,f64,str,str,str
241376,2015-10-25,2015-10-31,58,"""Purple Swamphen""","""Porphyrio porphyrio""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS""","""Atlas"""
241376,2015-10-25,2015-10-31,137,"""Pacific Golden Plover""","""Pluvialis fulva""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS""","""Atlas"""
241376,2015-10-25,2015-10-31,129,"""Ruddy Turnstone""","""Arenaria interpres""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS""","""Atlas"""
241376,2015-10-25,2015-10-31,153,"""Bar-tailed Godwit""","""Limosa lapponica""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS""","""Atlas"""
241376,2015-10-25,2015-10-31,124,"""Black Noddy""","""Anous minutus""",-31.583333,159.083333,-31.583333,159.083333,"""Lord Howe Island""","""TS""","""Atlas"""
…,…,…,…,…,…,…,…,…,…,…,…,…
902577,2022-08-21,2022-08-21,9,"""Stubble Quail""","""Coturnix pectoralis""",-31.916667,149.416667,-31.916667,149.416667,"""Moggymill""","""WS""","""DES"""
902577,2022-08-21,2022-08-21,705,"""Australian Magpie""","""Gymnorhina tibicen""",-31.916667,149.416667,-31.916667,149.416667,"""Moggymill""","""WS""","""DES"""
902577,2022-08-21,2022-08-21,364,"""Willie Wagtail""","""Rhipidura leucophrys""",-31.916667,149.416667,-31.916667,149.416667,"""Moggymill""","""WS""","""DES"""


In [1]:
all_obs.select(
    pl.col("Observation End Date").min().alias("min"),
    pl.col("Observation End Date").max().alias("max"),
)


NameError: name 'all_obs' is not defined